## USGS 3DEP DEM Generator Using the 1 km^2 GEOJSON Bounding Boxes

In [ ]:
# import packages
import os
import json
import geopandas as gpd
import pystac_client
import planetary_computer
import stackstac
from datetime import datetime
import xarray as xr
import rasterio
import rioxarray as rio 

### Define USGS 3DEP DEM downloader function

*** This was adapted from Eric Gagliano's Sentinel-1 stack downloader function ***
reference code can be found here: https://github.com/egagli/sar_snowmelt_timing/blob/main/sar_snowmelt_timing/s1_rtc_bs_utils.py

In [ ]:
# Define DEM downloader function
def get_usgs3dep_dem(bbox_gdf, start_time='1925-01-01', end_time='2020-05-06'):
    '''
    Returns a USGS 3DEP DEM xarray dataset using STAC data from Planetary Computer over the given time and bounding box.

            Parameters:
                    bbox_gdf (geopandas GeoDataFrame): geodataframe bounding box
                    start_time (str): start time of returned data 'YYYY-MM-DD' - keep this static at 1925-01-01
                    end_time (str): end time of returned data 'YYYY-MM-DD' - keep this static at 2020-05-06

            Returns:
                    stack (xarray dataset): xarray stack of all scenes in the specified spatio-temporal window

    '''
    
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    bbox = bbox_gdf.total_bounds
    search = catalog.search(
        collections=["3dep-seamless"],
        bbox=bbox,
        datetime=f"{start_time}/{end_time}",
        limit=1000
    )
    items = search.item_collection()

    # Stack the DEM
    stack = stackstac.stack(items, bounds_latlon=bbox, epsg=32610, dtype='float32', chunksize=512)

    return stack

### Define necessary folders:

In [ ]:
# Define folders where GeoJSONs and DEMs live/will live
geojson_folder = '../import/geojsons'
dem_folder = '../import/USGS3DEP_DEM'

os.makedirs(dem_folder, exist_ok=True)

In [ ]:
# Loop through each GeoJSON and download USGS 3DEP DEM
for filename in os.listdir(geojson_folder):
    if filename.endswith('.geojson'):
        geojson_path = os.path.join(geojson_folder, filename)
        bbox_gdf = gpd.read_file(geojson_path)

        # Get DEM for bounding box
        dem_scenes = get_usgs3dep_dem(bbox_gdf, start_time='1925-01-01', end_time='2020-05-06')

        # Print the dimensions - troubleshooting 
        #print(f"Dimensions of dem_scenes for {filename}: {dem_scenes.dims}")

        # Check file dimensions and select first band
        if len(dem_scenes.dims) >= 2:  # Ensure there are at least 2 dimensions (band, y, x)
            band_size = dem_scenes.shape[1]
            if band_size > 1:  # Check if there is more than one band
                dem_scenes = dem_scenes.isel(band=0)  # Select the first band

        if len(dem_scenes.dims) > 3:  # Ensure there is time dimension
            dem_scenes = dem_scenes.isel(time=0)

        # Save DEM as a GeoTIFF
        dem_filename = os.path.join(dem_folder, f"{os.path.splitext(filename)[0]}_3DEP_DEM.tif")
        dem_scenes.rio.to_raster(dem_filename) 
        print(f"Saved 3DEP DEM to {dem_filename}")